In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import features

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    df_cands = "train_carts_orders_w_features_new_split.parquet"
    # df_cands = "train_orders_new_split.parquet"
    new_split = "_new_split"


config = Config()

In [4]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=False, new_split=True)
# X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
#     config, 
#     X_train, 
#     X_val, 
#     X_val_seq, 
#     ordinal_encoder, 
#     return_only_val=True, 
#     truncate_val=True
# )
pad = ordinal_encoder.unknown_value

CPU times: user 15min 13s, sys: 2min 41s, total: 17min 55s
Wall time: 1min 46s


In [5]:
X_train = X_train.with_columns(
    [
        pl.col("user").cast(pl.UInt32, strict=False).alias("user"),
        pl.col("aid").cast(pl.UInt32, strict=False).alias("aid"),
    ]
)

X_val = X_val.with_columns(
    [
        pl.col("user").cast(pl.UInt32, strict=False).alias("user"),
        pl.col("aid").cast(pl.UInt32, strict=False).alias("aid"),
    ]
)

X_val_seq = X_val_seq.with_columns(
    [
        pl.col("user").cast(pl.UInt32, strict=False).alias("user"),
        pl.col("aid").cast(pl.UInt32, strict=False).alias("aid"),
    ]
)

In [6]:
del X_val
gc.collect()

1084

In [7]:
X_train.head()

user,aid,ts,type,n_session,num_action,n_session_right
u32,u32,i64,str,i32,i32,f64
14571581,1100210,1662328791563,"""clicks""",0,1,null
14571580,202353,1662328781067,"""clicks""",0,1,null
14571579,739876,1662328775605,"""clicks""",0,1,null
14571578,519105,1662328775009,"""clicks""",0,1,null
14571577,1141710,1662328774770,"""clicks""",0,1,null


In [8]:
# X_train = X_train.drop("n_session_right")

In [9]:
# df_full = pl.concat([X_train[["user", "aid", "ts"]], X_val_seq[["user", "aid", "ts"]]])

In [10]:
%%time

X_val_seq = X_val_seq.to_pandas()
X_val_seq = utils.make_date_stats(X_val_seq)
X_val_seq = X_val_seq.drop('date_day', axis=1)
X_val_seq = pl.from_pandas(X_val_seq)

CPU times: user 2.48 s, sys: 389 ms, total: 2.87 s
Wall time: 2.74 s


In [11]:
# %%time

# X_train = X_train.to_pandas()
# X_train = utils.make_date_stats(X_train)
# X_train = X_train.drop('date_day', axis=1)
# X_train = pl.from_pandas(X_train)

In [12]:
X_val_seq.head()

user,aid,ts,type,n_session,num_action,n_session_right,date,n_day,n_week,dayofweek,hour
u32,u32,i64,str,i32,i32,f64,datetime[ns],i8,i8,i8,i8
14571581,1100210,1662328791563,"""clicks""",0,1,null,2022-09-04 23:59:51.563000064,4,35,6,23
14571580,202353,1662328781067,"""clicks""",0,1,null,2022-09-04 23:59:41.067000064,4,35,6,23
14571579,739876,1662328775605,"""clicks""",0,1,null,2022-09-04 23:59:35.604999936,4,35,6,23
14571578,519105,1662328775009,"""clicks""",0,1,null,2022-09-04 23:59:35.008999936,4,35,6,23
14571577,1141710,1662328774770,"""clicks""",0,1,null,2022-09-04 23:59:34.769999872,4,35,6,23


In [13]:
X_val_seq.shape

(5072769, 12)

In [14]:
X_val_seq = X_val_seq.rename({"user": "session"})
X_val_seq = X_val_seq.rename({"aid": "item_id"})

X_train = X_train.rename({"user": "session"})
X_train = X_train.rename({"aid": "item_id"})

In [15]:
X_val_seq.tail()

session,item_id,ts,type,n_session,num_action,n_session_right,date,n_day,n_week,dayofweek,hour
u32,u32,i64,str,i32,i32,f64,datetime[ns],i8,i8,i8,i8
12899783,255297,1661724000572,"""clicks""",0,11,null,2022-08-29 00:00:00.572,29,35,0,0
12899782,1669402,1661724000568,"""clicks""",0,70,null,2022-08-29 00:00:00.568,29,35,0,0
12899781,141736,1661724000559,"""clicks""",0,11,null,2022-08-29 00:00:00.559000064,29,35,0,0
12899780,1142000,1661724000378,"""clicks""",0,5,null,2022-08-29 00:00:00.377999872,29,35,0,0
12899779,59625,1661724000278,"""clicks""",0,1,null,2022-08-29 00:00:00.278000128,29,35,0,0


In [16]:
col_user = "session"
col_item = "item_id"

In [17]:
X_val_seq.filter(pl.col(col_user) == 14052544)

session,item_id,ts,type,n_session,num_action,n_session_right,date,n_day,n_week,dayofweek,hour
u32,u32,i64,str,i32,i32,f64,datetime[ns],i8,i8,i8,i8
14052544,1233376,1662264288865,"""clicks""",1,4,null,2022-09-04 06:04:48.864999936,4,35,6,6
14052544,1143465,1662264165486,"""clicks""",1,5,null,2022-09-04 06:02:45.486000128,4,35,6,6
14052544,1611515,1662263333401,"""clicks""",1,6,null,2022-09-04 05:48:53.400999936,4,35,6,5
14052544,1611515,1662134829403,"""clicks""",0,7,null,2022-09-02 18:07:09.403000064,2,35,4,18
14052544,1402459,1662134499406,"""clicks""",0,8,null,2022-09-02 18:01:39.406000128,2,35,4,18
14052544,313976,1662134495799,"""clicks""",0,9,null,2022-09-02 18:01:35.799000064,2,35,4,18
14052544,1402459,1662134434310,"""clicks""",0,10,null,2022-09-02 18:00:34.310000128,2,35,4,18


In [18]:
X_train.head()

session,item_id,ts,type,n_session,num_action,n_session_right
u32,u32,i64,str,i32,i32,f64
14571581,1100210,1662328791563,"""clicks""",0,1,null
14571580,202353,1662328781067,"""clicks""",0,1,null
14571579,739876,1662328775605,"""clicks""",0,1,null
14571578,519105,1662328775009,"""clicks""",0,1,null
14571577,1141710,1662328774770,"""clicks""",0,1,null


In [19]:
X_val_seq.head()

session,item_id,ts,type,n_session,num_action,n_session_right,date,n_day,n_week,dayofweek,hour
u32,u32,i64,str,i32,i32,f64,datetime[ns],i8,i8,i8,i8
14571581,1100210,1662328791563,"""clicks""",0,1,null,2022-09-04 23:59:51.563000064,4,35,6,23
14571580,202353,1662328781067,"""clicks""",0,1,null,2022-09-04 23:59:41.067000064,4,35,6,23
14571579,739876,1662328775605,"""clicks""",0,1,null,2022-09-04 23:59:35.604999936,4,35,6,23
14571578,519105,1662328775009,"""clicks""",0,1,null,2022-09-04 23:59:35.008999936,4,35,6,23
14571577,1141710,1662328774770,"""clicks""",0,1,null,2022-09-04 23:59:34.769999872,4,35,6,23


In [20]:
X_val_seq.shape

(5072769, 12)

In [21]:
%%time

df = pl.read_parquet(config.cur_model_output_path + config.df_cands)

CPU times: user 25.1 s, sys: 13.7 s, total: 38.8 s
Wall time: 5.12 s


In [22]:
%%time

df = features.make_user_features(df, X_val_seq)

CPU times: user 14.4 s, sys: 8.05 s, total: 22.5 s
Wall time: 4.64 s


In [23]:
cols_rank = [col for col in df.columns if col.endswith("_rank")]
cols_rank

['rnn_rank',
 'weak_nns_rank',
 'cooc_rank',
 'bm25_rank',
 'tfidf_rank',
 'cosine_rank',
 'last_items_rank']

In [24]:
%%time

df = features.make_inverse_rank_features(df)

CPU times: user 2.76 s, sys: 2.76 s, total: 5.51 s
Wall time: 4.91 s


In [25]:
df.shape

(50789844, 73)

In [26]:
df.head()

item_id,session,rnn_score,rnn_rank,weak_nns_score,weak_nns_rank,cooc_score,cooc_rank,bm25_score,bm25_rank,tfidf_score,tfidf_rank,cosine_score,cosine_rank,last_items_rank,target,rank_avg,rank_min,rank_max,rank_std,rank_median,rank_cnt_nulls,ratio_clicks_to_carts,ratio_clicks_to_orders,ratio_carts_to_orders,ratio_clicks_to_carts_test_week,ratio_clicks_to_orders_test_week,ratio_carts_to_orders_test_week,time_since_last_clicks_this_item_by_user,time_since_last_carts_this_item_by_user,time_since_last_orders_this_item_by_user,time_since_last_interaction_this_item_by_user,last_interaction_type,cnt_interactions_total,cnt_interactions_clicks,cnt_interactions_carts,cnt_interactions_orders,ratio_user_clicks_to_carts,ratio_user_clicks_to_orders,ratio_user_carts_to_orders,item_popularity,item_popularity_total_mean,item_popularity_total_median,item_popularity_total_std,item_popularity_total_min,item_popularity_total_max,item_popularity_to_item_popularity_total_mean,item_popularity_to_item_popularity_total_median,item_popularity_clicks_mean,item_popularity_clicks_median,item_popularity_clicks_std,item_popularity_clicks_min,item_popularity_clicks_max,item_popularity_to_item_popularity_clicks_mean,item_popularity_to_item_popularity_clicks_median,item_popularity_carts_mean,item_popularity_carts_median,item_popularity_carts_std,item_popularity_carts_min,item_popularity_carts_max,item_popularity_to_item_popularity_carts_mean,item_popularity_to_item_popularity_carts_median,item_popularity_orders_mean,item_popularity_orders_median,item_popularity_orders_std,item_popularity_orders_min,item_popularity_orders_max,item_popularity_to_item_popularity_orders_mean,item_popularity_to_item_popularity_orders_median,ratio_user_total_clicks_to_carts,ratio_user_total_clicks_to_orders,ratio_user_total_carts_to_orders,inverse_rank_level_1
u32,u32,f32,i8,i32,i8,f32,i8,f32,i8,f32,i8,f32,i8,i8,u8,f32,i8,i8,f32,f32,i8,f32,f32,f32,f32,f32,f32,u32,u32,u32,u32,u8,u8,u8,u8,u8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
1007613,12899782,0.123797,1,null,-99,0.428216,1,9459.402344,1,0.446578,1,0.446578,1,1,0,1.0,1,1,0.0,1.0,1,0.112676,0.053853,0.477941,0.058824,0.009804,0.166667,null,null,0,0,2,1,0,0,1,NaN,inf,inf,4.835381,5.257325,2.09828,10.757962,0.022113,73.570023,0.919742,2.30445,3.588506,1.808354,4.722032,0.022113,21.054054,1.347464,2.673913,3.926136,2.09828,5.293334,0.022113,21.054054,1.231588,2.30445,24.87027,4.835381,32.151772,1.601966,73.570023,0.194424,1.0,0.347826,0.108696,0.3125,6.0
409367,12899782,0.020285,2,null,-99,null,-99,219.458786,73,0.017732,47,null,-99,-99,0,40.666668,2,73,35.921211,47.0,4,0.167594,0.075896,0.452855,0.079038,0.017182,0.217391,null,null,null,null,null,0,0,0,0,NaN,NaN,NaN,20.700245,5.257325,2.09828,10.757962,0.022113,73.570023,3.93741,9.865339,3.588506,1.808354,4.722032,0.022113,21.054054,5.768486,11.447011,3.926136,2.09828,5.293334,0.022113,21.054054,5.272421,9.865339,24.87027,4.835381,32.151772,1.601966,73.570023,0.832329,4.280996,0.347826,0.108696,0.3125,0.534975
383474,12899782,0.019381,3,null,-99,null,-99,861.626038,11,0.062752,6,0.059899,5,-99,0,6.25,3,11,3.40343,5.5,3,0.128254,0.054692,0.426431,0.109589,0.006849,0.0625,null,null,null,null,null,0,0,0,0,NaN,NaN,NaN,24.336609,5.257325,2.09828,10.757962,0.022113,73.570023,4.629086,11.598361,3.588506,1.808354,4.722032,0.022113,21.054054,6.781823,13.45788,3.926136,2.09828,5.293334,0.022113,21.054054,6.198616,11.598361,24.87027,4.835381,32.151772,1.601966,73.570023,0.978542,5.033029,0.347826,0.108696,0.3125,0.790909
1779730,12899782,0.013374,4,null,-99,null,-99,441.777557,33,0.025824,23,0.017524,68,-99,0,32.0,4,68,26.845236,28.0,3,0.056959,0.023643,0.415094,0.081633,0.020408,0.25,null,null,null,null,null,0,0,0,0,NaN,NaN,NaN,6.766585,5.257325,2.09828,10.757962,0.022113,73.570023,1.287077,3.224824,3.588506,1.808354,4.722032,0.022113,21.054054,1.885628,3.741848,3.926136,2.09828,5.293334,0.0221

In [27]:
df = df.sort(col_user)

In [28]:
%%time

df.write_parquet(config.cur_model_output_path + f'final_train_carts_orders_w_features{config.new_split}.parquet')

CPU times: user 38.7 s, sys: 20.2 s, total: 58.9 s
Wall time: 31.4 s


In [29]:
stop

NameError: name 'stop' is not defined

In [ ]:
df_target_stats = df.groupby("session").agg(pl.col("target").sum())
df_target_stats = df_target_stats.filter(pl.col("target") > 0)
df = df.filter(pl.col("session").is_in(df_target_stats["session"]))

del df_target_stats
gc.collect()

In [ ]:
%%time

df = df.with_column(((pl.col("target") > 0) * 1).alias("target")).with_column(
    pl.col("target").cast(pl.UInt8, strict=False).alias("target"),
)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
%%time

df = features.make_rank_features(df)

In [ ]:
%%time

df = features.make_item_interaction_to_interaction_features(df, X_train)

In [ ]:
df.head()

In [ ]:
df.filter(pl.col("ratio_carts_to_orders") > 0)

In [ ]:
%%time

df = features.make_item_last_interation_features(df, X_val_seq)

In [ ]:
df.head()

In [ ]:
df.filter(pl.col("time_since_last_interaction_this_item_by_user") > 0).head()

In [ ]:
%%time

dict_df_history_cnt_interactions_types = features.get_user_item_stats(X_val_seq)

In [ ]:
%%time

df = features.make_user_item_features(df, dict_df_history_cnt_interactions_types)

In [ ]:
del dict_df_history_cnt_interactions_types
gc.collect()

In [ ]:
%%time

item_popularity_stats = features.get_item_popularity_stats(X_train, col_item=col_item)

In [ ]:
del X_train
gc.collect()

In [ ]:
%%time

dict_user_item_popularity_stats = features.get_user_item_popularity_stats(X_val_seq, item_popularity_stats)

In [ ]:
del X_val_seq
gc.collect()

In [ ]:
# del df_full
# gc.collect()

In [ ]:
%%time

df = features.make_user_item_popularity_features(df, item_popularity_stats, dict_user_item_popularity_stats)

In [ ]:
del item_popularity_stats, dict_user_item_popularity_stats
gc.collect()

In [ ]:
col_user = "session"
col_item = "item_id"

In [ ]:
%%time

cols_rank = [col for col in df.columns if col.endswith('_rank')]
cols_rank.extend(['rank_min', 'rank_max', 'rank_cnt_nulls'])
for col in cols_rank:
    df = df.with_column(pl.col(col).fill_null(-99))
    df = df.with_column(
        pl.col(col).cast(pl.Int8, strict=False).alias(col)
    )

In [ ]:
df

In [ ]:
df = df.sort(col_user)

In [ ]:
%%time

df.write_parquet(config.cur_model_output_path + f'train_carts_orders_w_features{config.new_split}.parquet')

In [ ]:
# %%time

# df.write_parquet(config.cur_model_output_path + f'train_orders_w_features{config.new_split}.parquet')

In [ ]:
len(df[col_user].unique())

In [ ]:
stop

In [ ]:
df = pl.read_parquet(
    config.cur_model_output_path
    + f"train_carts_orders_w_features{config.new_split}.parquet"
)

In [ ]:
len(df["session"].unique())

In [ ]:
X_val.shape

In [ ]:
X_val = X_val.filter(pl.col("type") != "clicks")

In [ ]:
X_val.shape

In [ ]:
X_val

In [ ]:
mod = "val"

In [ ]:
%%time

dict_models_preds = dict()

cols_to_drop = ['index']

cols_to_rename = dict()
cols_to_rename[config.col_user] = 'session'
cols_to_rename['item_id'] = 'preds'

cols_to_add_model_name = ['preds']

model_names = ['rnn', 'cooc_smart', 'bm25', 'tfidf', 'cosine', 'last_items']

for model_name in model_names:
    dict_models_preds[model_name] = pl.read_parquet(config.cur_model_output_path + f'cur_{model_name}_df_{mod}_preds_100.parquet')
    for col in cols_to_drop:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].drop(col)
    for col in cols_to_rename:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: cols_to_rename[col]})
    for col in cols_to_add_model_name:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: model_name + '_' + col})
    if model_name == 'rnn':
        dict_models_preds[model_name] = dict_models_preds[model_name].rename({'preds_scores': 'rnn_scores'})
        dict_models_preds[model_name] = dict_models_preds[model_name].with_column(
            pl.col("rnn_scores")
                .rank(method="ordinal", reverse=True)
                .over(["session"])
                .alias("rnn_rank")
        )
    elif model_name in ['cooc_smart']:
        for col in ['cooc_triplet_score_0', 'cooc_triplet_rank_0', 'cooc_score_0', 'cooc_rank_0']:
            if col in dict_models_preds[model_name]:
                dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: col[:-2]})
    for col in dict_models_preds[model_name].columns:
        if 'preds' in col:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: 'item_id'})

In [ ]:
dict_models_preds["bm25"].head()

In [ ]:
dict_models_preds["last_items"].head()

In [ ]:
model_names

In [ ]:
%%time

df_merged = dict_models_preds[model_names[0]]
for i in tqdm(range(1, len(model_names))):
    model_name_cur = model_names[i]
    df_merged = utils.polars_outer_merge(df_merged, dict_models_preds[model_name_cur])

In [ ]:
%%time

X_val_lists = X_val.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

In [ ]:
%%time

df_targets = utils.create_targets_df(list(X_val_lists.to_pandas()[config.col_train_seq]))

In [ ]:
%%time

df_users = pl.from_pandas(X_val_lists.to_pandas()[[config.col_user]].reset_index())
df_users = df_users.join(df_targets, on='index', how='inner')
df_users = df_users.drop('index')
df_users = df_users.rename({config.col_user: 'session'})

In [ ]:
%%time

df_merged = utils.polars_outer_merge(df_merged, df_users)

In [ ]:
del df_users, dict_models_preds
gc.collect()

In [ ]:
columns_from_models = [
    col for col in df_merged.columns if "_score" in col or "_rank" in col
]

In [ ]:
# %%time

# df_merged = pl.from_pandas(df_merged.to_pandas().dropna(subset=columns_from_models, thresh=1))

In [ ]:
%%time

df_merged = df_merged.with_column(pl.col("target").fill_null(0))
df_merged = df_merged.with_columns(
        [
            pl.col('target')
            .cast(pl.Float32)
            .alias('target'),
        ]
    )

In [ ]:
df_merged.head()

In [ ]:
df_merged.shape

In [ ]:
df_merged = df_merged.sort("session")

In [ ]:
df_merged.head()

In [ ]:
neg_sample_size = len(df_merged.filter(pl.col("target") > 0)) * 20
df_merged = pl.concat(
    [
        df_merged.filter(pl.col("target") > 0),
        df_merged.filter(pl.col("target") == 0).sample(neg_sample_size, seed=0),
    ]
)

In [ ]:
df_merged = df_merged.sort("session")

In [ ]:
%%time

df_merged.write_parquet(config.cur_model_output_path + 'train_clicks_orders.parquet')

In [ ]:
stop

In [ ]:
qids_train = df_merged.to_pandas().groupby(["session"])["item_id"].count().values

In [ ]:
qids_train

In [ ]:
from lightgbm import LGBMRanker

In [ ]:
df_merged

In [ ]:
def find_query(q: str, data: list) -> list:
    path_list = list()

    def helper(subdata, path):
        for index, item in enumerate(subdata):
            if isinstance(item, list):
                helper(item, path + [index])
            elif item == q:
                path_list.append(path + [index])

    helper(data, list())
    return path_list

In [ ]:
data = ["a", ["b", "c", ["d", "a"]]]

In [ ]:
find_query("d", data)

In [ ]:
import lightgbm as lgb

In [ ]:
bst = lgb.Booster(model_file="../src/otto_recommender/booster_fold2.lgb")

In [ ]:
importance_df = (
    pd.DataFrame(
        {
            "feature_name": bst.feature_name(),
            "importance_gain": bst.feature_importance(importance_type="gain"),
            "importance_split": bst.feature_importance(importance_type="split"),
        }
    )
    .sort_values("importance_gain", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
importance_df

In [ ]:
lgb.plot_importance(bst, importance_type="gain")

In [ ]:
df_merged = pl.read_parquet(
    config.cur_model_output_path + "train_clicks_orders.parquet"
).to_pandas()

In [ ]:
df_merged["session"].nunique()

In [ ]:
df_merged[df_merged["session"] == 13405261]

In [ ]:
df_merged

In [ ]:
%%time

columns = [
    col for col in df_merged.columns if "_score" in col or "_rank" in col
]
X_train = df_merged[columns].to_numpy(dtype=np.float32)
df_merged['target'] = (df_merged['target'] > 0) * 1
y_train = df_merged['target'].astype(np.int8)

In [ ]:
%%time

train_groups = utils.get_group_for_lgb(df_merged["session"].values)

In [ ]:
%%time

lgb_train = lgb.Dataset(
    X_train,
    y_train,
    feature_name=columns,
    group=train_groups,
    free_raw_data=False,
)

In [ ]:
%%time

lgb_train.construct()

In [ ]:
lgb_train.get_label()

In [ ]:
lgb_train.get_feature_name()

In [ ]:
len(lgb_train.get_data())

In [ ]:
%%time

len(np.where((np.isnan(lgb_train.get_data()).all(axis=1) == False) & (lgb_train.get_label() == 1))[0])

In [ ]:
df_merged.loc[
    np.where(
        (np.isnan(lgb_train.get_data()).all(axis=1) == False)
        & (lgb_train.get_label() == 1)
    )[0]
]

In [ ]:
946438 / len(df_merged[df_merged["target"] == 1])

In [ ]:
df_merged.loc[
    np.where(
        (np.isnan(lgb_train.get_data()).all(axis=1) == True)
        & (lgb_train.get_label() == 0)
    )[0]
]